# 演習1. スレッドとパイプライン

**進め方**：上のセルから順に ▶ を押すだけです。
コードセルの1行目の `%%writefile` は「このセルの中身をファイルとして保存する」という
Colab の命令で、次のセルでコンパイル・実行します。

この演習で答えるのは、**「なぜ分担すると速くなるのか」**の1点だけです。

> **予測クイズは、必ず実行前に自分で予測**してください。予測が外れることに意味があります。

## 1-0. 言葉の整理

- **プロセス**：OS がプログラムを実行する単位。`./a.out` と打つたびに1つ生まれる
- **スレッド**：プロセスの中にある「実行の流れ」。プロセスが生まれた時点で `main()` を実行する
  スレッドが1本入っている。`std::thread` はこの流れを**増やす**道具

```
プロセス（自分のプログラム）        ← ./a.out で生まれる
  ├─ スレッド ⇒ main() を実行する（最初からある）
  ├─ スレッド ⇒ std::thread で増やした
  └─ スレッド ⇒ std::thread で増やした
```

**なぜプロセスではなくスレッドか。** メモリの扱いが違うからです。

| | プロセス | スレッド |
|---|---|---|
| メモリ | 各自が専用。共有には仕組みの作り込みが要る | **何もしなくても共有** |
| データの受け渡し | 通信・共有メモリなどを設計する | **同じ変数を直接見るだけ** |
| 作る手間 | 重い | 軽い |
| 相手を壊せるか | 壊しにくい | **簡単に壊せる** |

画像1枚は数百KB〜数MBあります。これを頻繁に受け渡すプログラムでは、
「何もしなくても同じメモリが見えている」手軽さが効きます。

**ただしこの長所は、そのまま短所です。** 同じ変数に2つのスレッドが同時に触れば値が壊れます。
だから並行処理の話は、いつも**「速くする話」と「壊さない話」**がセットになります。
この演習1は前者、演習2からが後者です。

## 1-1. 分担のしかたは2通りある

### ケース1：仕事どうしが独立 ⇒ そのまま重ねる（**並列**）

```
【1人で順番に】                 【2スレッドで】
   0    10   20 (分)               0    10 (分)
A  AAAAA.....                   A  AAAAA
B  .....BBBBB                   B  BBBBB
   合計 20分                       合計 10分
```

（`A` `B` は働いている時間、`.` は何もしていない時間）

### ケース2：1つの仕事が「段」の連なり ⇒ ずらして重ねる（**パイプライン**）

「読む → 処理する → 出す」には順序があるので、同じ1件の Read と Infer は同時にできません。
それでも**件をまたげば重ねられます**。

```
1文字 = 5ms、数字はフレーム番号、各段 30ms

【1人で順番に】                          【3人で分担（パイプライン）】
Read   111111............222222......    Read   111111222222333333444444
Infer  ......111111............222222    Infer  ......111111222222333333
Show   ............111111............    Show   ............111111222222
```

**「行がそろって重なる＝並列」「ずれて重なる＝パイプライン」**。この2つの絵を区別してください。

### 速さの物差しは2つある

- **レイテンシ**：**1件**が入ってから出るまでの時間（例：1フレーム 90ms）
- **スループット**：単位時間に**何件**さばけるか（例：33 FPS ―― **FPS はスループット**）

> **スレッドは1件を速くしない。同じ時間に多くの件をさばけるようにする。**
> 改善するのは**スループット**であって、レイテンシではありません。

動画処理で上げたい FPS はスループットなので、スレッドが効きます。

## 1-2. 【予測クイズ】独立な仕事を2スレッドに分ける

**仕事A**（300ms）と**仕事B**（300ms）を片づけます。互いに無関係なので順序は問いません。

- **1スレッドで**：1本が 仕事A → 仕事B の順にこなす
- **2スレッドで**：スレッド1が仕事A、スレッド2が仕事B

実行すると、**どのスレッドがいつ働いていたか**がタイムチャートで出ます。
**1行が1本のスレッド**、行の中の `A` `B` がどちらの仕事をしているかです。

**実行前に予測してください。それぞれ何 ms かかるでしょうか。**

In [ ]:
%%writefile ex01a.cpp
#include <iostream>
#include <string>
#include <algorithm>
#include <thread>
#include <chrono>
using namespace std;
using namespace std::chrono;

const int SCALE = 20;                    // 図の1文字あたりの時間(ms)
steady_clock::time_point t0;
int t_begin[2], t_end[2];                // 仕事A(=0) と 仕事B(=1) の開始/終了時刻
const char MARK[2] = {'A', 'B'};

int now_ms() { return (int)duration_cast<milliseconds>(steady_clock::now() - t0).count(); }

// 300ms かかる仕事を1つこなす（id=0 が仕事A、id=1 が仕事B）
void job(int id) {
    t_begin[id] = now_ms();
    this_thread::sleep_for(milliseconds(300));
    t_end[id] = now_ms();
}
// ※ 実行中は何も表示しない。開始・終了時刻だけを記録し、最後に draw() がまとめて図を描く。
//    （複数のスレッドが同時に画面へ書き込むと表示が乱れることがあるため。詳しくは演習2で）

void draw(bool oneThread) {
    int last = max(t_end[0], t_end[1]);
    int w = last / SCALE;

    if (oneThread) {
        // スレッドは1本。その1本が仕事A→仕事Bの順にこなす
        string s(w, '.');
        for (int id = 0; id < 2; id++)
            for (int c = t_begin[id] / SCALE; c < t_end[id] / SCALE && c < w; c++) s[c] = MARK[id];
        cout << "T1   " << s << "\n";
        cout << "Total " << last << " ms\n";
    } else {
        for (int id = 0; id < 2; id++) {
            string s(w, '.');
            for (int c = t_begin[id] / SCALE; c < t_end[id] / SCALE && c < w; c++) s[c] = MARK[id];
            cout << "T" << (id + 1) << "   " << s << "\n";
        }
        int ov = min(t_end[0], t_end[1]) - max(t_begin[0], t_begin[1]);
        if (ov < 0) ov = 0;
        cout << "Total " << last << " ms   （T1 と T2 が同時に働いた時間 = " << ov << " ms）\n";
    }
}

int main() {
    cout << "図の見かた： 1文字 = 20ms（帯の長さがそのまま所要時間）\n"
         << "  T1 / T2 = スレッド（行が1本 = スレッド1本）\n"
         << "  A = 仕事A（300ms）、B = 仕事B（300ms）、'.' = 何もしていない\n";

    cout << "\n【1スレッドで順番に】   スレッドは1本だけ\n";
    t0 = steady_clock::now();
    job(0);
    job(1);
    draw(true);

    cout << "\n【2スレッドで】   スレッドは2本\n";
    t0 = steady_clock::now();
    thread t1(job, 0);
    thread t2(job, 1);
    t1.join();
    t2.join();
    draw(false);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex01a.cpp -o ex01a && ./ex01a

### 結果 ―― 600ms が 300ms に

```
【1スレッドで順番に】
T1   AAAAAAAAAAAAAAABBBBBBBBBBBBBBB
Total 600 ms

【2スレッドで】
T1   AAAAAAAAAAAAAAA
T2   BBBBBBBBBBBBBBB
Total 300 ms
```

2本の行が**どちらも左端（0ms）から始まっている** ―― これが「同時に動いている」ということです。
仕事A と 仕事B には順序がないので、ずれずにまるごと重なります。**ケース1（並列）の絵**です。

**レイテンシは変わっていません。** 仕事A はどちらも 300ms のままです。
変わったのはスループット（600ms で2件 → 300ms で2件）だけです。

### 覚えるべき3つ

```cpp
std::thread t1(job, 0);   // ① スレッドを作る（第1引数が関数、あとはその関数への引数）
t1.join();                // ② そのスレッドが終わるまで待つ
```
```bash
g++ -std=c++17 -pthread ex01a.cpp -o ex01a    # ③ -pthread が要る
```

- **`join()` を忘れる**と `main` が先に終わり、プログラムが異常終了します（発展課題1-2）
- スレッドに**参照を渡すには `std::ref(変数)`**（C/C++問題集 問9）

## 1-3. 【予測クイズ】人数を増やせば増やすほど速くなるか

性質の違う2種類の仕事を比べます。

- **待つ仕事** … 300ms 待つだけ（ファイル読み込みや外部ハードウェアの応答待ちのモデル）
- **計算する仕事** … CPU で計算し続ける（およそ300ms分）

それぞれ4件を、1人・2人・4人で分担します。**どちらも人数に比例して速くなるでしょうか。**

In [ ]:
%%writefile ex01b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
using namespace std::chrono;

// 待つ仕事：300ms 待つだけ（ファイル読み込みや外部ハードウェアの応答待ちのモデル）
void wait_job(int) { std::this_thread::sleep_for(milliseconds(300)); }

// 計算する仕事：CPU で計算し続ける（およそ300ms分）
long calc_job(int seed) {
    long s = 0;
    for (int i = 0; i < 100000000; i++) s += (seed + i) % 7;
    return s;
}

template <class Job>
void run(const char* kind, Job job, int nthread, int njob) {
    auto t0 = steady_clock::now();
    std::vector<std::thread> ts;
    for (int k = 0; k < nthread; k++)
        ts.emplace_back([=]() {          // k 番目のスレッドは k, k+n, k+2n ... 件目を担当
            for (int j = k; j < njob; j += nthread) job(j);
        });
    for (auto& t : ts) t.join();
    std::cout << kind << " " << nthread << " スレッド : "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
}

int main() {
    std::cout << "hardware_concurrency() = "
              << std::thread::hardware_concurrency() << "\n\n";
    for (int n : {1, 2, 4}) run("待つ仕事    ", wait_job, n, 4);
    std::cout << "\n";
    for (int n : {1, 2, 4}) run("計算する仕事", calc_job, n, 4);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex01b.cpp -o ex01b && ./ex01b

### 結果 ―― 待つ仕事だけが人数に比例する

```
待つ仕事     1 スレッド : 1200 ms
待つ仕事     2 スレッド :  600 ms   ← 人数分だけ速くなる
待つ仕事     4 スレッド :  300 ms   ←

計算する仕事（Colab の典型例）      参考：計算回路を4つ積んだPC
1 スレッド : 1373 ms                1 スレッド : 1300 ms
2 スレッド : 1356 ms                2 スレッド :  660 ms  ← 半分
4 スレッド : 1364 ms                4 スレッド :  340 ms  ← さらに半分
   ↑ 何スレッドにしても変わらない
```

- **待つ仕事**は、待つのに計算回路が要らないので、**何人でも同時に待てます**
- **計算する仕事**は、**「同時に計算できる数」で頭打ち**になります

Colab の「2コア」は、実は**計算回路1つ分の性能しかありません**。
`hardware_concurrency()` は 2 と答えるのに、計算は1人分しか速くならない ――
**この数字は目安であって、当てになりません**（演習3で測り直します）。

計算が中心の仕事を **CPUバウンド**、待ちが中心の仕事を **I/Oバウンド** と呼びます。

### これがパイプラインの土台

動画処理の各段は、どれも**「頼んで、待つ」**形です。

- **動画の読み込み** ⇒ データが届くまで CPU は手が空いている
- **アクセラレータへの処理依頼** ⇒ 計算するのは向こう側の回路。CPU は待つだけ
- **画面への表示** ⇒ 送ったら、描き終わるまで待つだけ

だから **計算回路の少ないマシンでも、スレッドの数だけ重ねられます。**

## 1-4. パイプラインとボトルネック

各段 30ms、3フレーム分。`R11111` は「**R**ead 係が**フレーム1**を処理中」の意味です。

```
【1スレッドで順番に】
T1    R11111I11111S11111R22222I22222S22222R33333I33333S33333
Total 270 ms   →  11.1 FPS

【3スレッドで（パイプライン）】
T1    R11111R22222R33333............     ← Read 係
T2    ......I11111I22222I33333......     ← Infer 係
T3    ............S11111S22222S33333     ← Show 係
                  ^^^^^^
                  この列では T1=フレーム3を Read、T2=2を Infer、T3=1を Show
Total 150 ms   →  20.0 FPS
```

**同じ瞬間に、3つの異なるフレームが3本のスレッドで進んでいます。** これがパイプラインです。
長く流し続ければ 1フレームあたり 30ms（一番遅い段の時間）に近づき、1スレッドの 90ms の約3倍です。

### 段の重さが違うと、そうはならない

Read=10ms、Infer=60ms、Show=10ms の場合。

```
【1スレッドで順番に】
T1    R1I11111111111S1R2I22222222222S2R3I33333333333S3
Total 240 ms   →  12.5 FPS

【3スレッドで】
T1    R1R2R3..................................     ← すぐ終わって手待ち
T2    ..I11111111111I22222222222I33333333333..     ← 働きづめ
T3    ..............S1..........S2..........S3     ← ほとんど手待ち
Total 200 ms   →  15.0 FPS   （1.2倍にしかならない）
```

3人に分担したのに、実際にはほぼ1人しか働いていません。
Read と Show をいくら速くしても全体は変わりません。

> **パイプラインの速さは、一番遅い段で決まる。この段を「ボトルネック」と呼ぶ。**

ここでもレイテンシは縮んでいません（1件 10+60+10=80ms のまま）。上がったのは FPS だけです。

> **細かいこと**：上の図で Read 係は最初にどんどん先へ進めてしまえます。実際そうすると、
> 読んだデータが行き場を失ってメモリに溜まります。段と段のあいだのキューに**容量の上限**を
> 設けてこれを防ぎます（演習2・3）。

## まとめ

- スレッドが改善するのは**スループット**。レイテンシは縮まない
- **待つ仕事**はスレッド数だけ重なる。**計算する仕事**は「同時に計算できる数」で頭打ち
- パイプラインの速さは**一番遅い段（ボトルネック）**で決まる

**次は演習2** ―― 段と段をつなぐ「キュー」の使い方です。

> 🧩 この演習の**発展課題**は `adv01_threads.ipynb` にあります（問題は README にも載せてあります）。
> 全部やる必要はありません。**自分のやりたいことに関係の深いものだけ**選んでください。